### Up–down symmetrization of SPARC

This example loads the SPARC active coils, passive structures, limiter and wall, then prepares an exactly up–down symmetric description about `Z = 0`. 


In [ ]:
from freegsnke import build_machine
import matplotlib.pyplot as plt

import pickle
from pathlib import Path

import numpy as np
import pytest
import os 

from freegsnke.build_machine import build_tokamak_components
from freegsnke.up_down_symmetry import prepare_up_down_symmetric_machine

REPO_ROOT = "/Users/bf3280/Documents/freegsnke"

In [ ]:
machine_config_dir = os.path.join(REPO_ROOT, "machine_configs", "SPARC")
print(machine_config_dir)
machine_data = {}
for component in ("active_coils", "passive_coils", "limiter", "wall"):
    path = os.path.join(machine_config_dir , f"SPARC_{component}.pickle")
    print(path)
    with open(path, "rb") as machine_file:
        machine_data[component] = pickle.load(machine_file)

passive_pairs = [
    (f"vacuum_vessel_{index}", f"vacuum_vessel_{index + 1}")
    for index in range(0, 16, 2)
]
passive_pairs.append(("VSC_coil_cover0", "VSC_coil_cover1"))
prepared = prepare_up_down_symmetric_machine(
    machine_data["active_coils"],
    machine_data["passive_coils"],
    machine_data["limiter"],
    machine_data["wall"],
    passive_pairs=passive_pairs,
    exclude_odd_active=True,
)

tokamak = build_machine.tokamak(
    active_coils_data=prepared.even_active_coils_data,
    passive_coils_data=prepared.passive_coils_data,
    limiter_data=prepared.limiter_data,
    wall_data=prepared.wall_data,
)       

_, ax1 = plt.subplots(1, 1, figsize=(4, 8), dpi=80)
ax1.grid(alpha=0.5)
ax1.set_aspect('equal')
ax1.set_xlim(0.1, 2.15)
ax1.set_ylim(-2.5, 2.5)
ax1.set_xlabel(r'Major radius, $R$ [m]')
ax1.set_ylabel(r'Height, $Z$ [m]')
tokamak.plot(axis=ax1, show=False)

assert len(prepared.passive_pairs) == len(passive_pairs)
assert prepared.excluded_odd_active_names == ("VS1",)


### Verification of passive coil symmetry

Each upper passive coil is verified to be symmetric with its corresponding lower coil about the midplane ($Z=0$).  
The lower coil coordinates are reflected as $(R,Z) \rightarrow (R,-Z)$, and the test verifies that every point of the upper coil has a matching reflected point within numerical tolerance. It also checks that the nearest-neighbour mapping is one-to-one.

In [16]:
for pair in prepared.passive_pairs:
    upper, lower = (
        next(
            element
            for element in prepared.passive_coils_data
            if element["name"] == name
        )
        for name in (pair.upper_name, pair.lower_name)
    )
    upper_points = np.column_stack((upper["R"], upper["Z"]))
    reflected_lower = np.column_stack((lower["R"], -np.asarray(lower["Z"])))

    distances = np.linalg.norm(
        upper_points[:, np.newaxis] - reflected_lower[np.newaxis, :],
        axis=-1,
    )

    nearest_indices = np.argmin(distances, axis=1)
    nearest_distances = np.min(distances, axis=1)

    np.testing.assert_allclose(nearest_distances, 0.0, atol=1e-14)

    # Every reflected lower point must be matched exactly once
    assert len(np.unique(nearest_indices)) == len(upper_points)

### Verification of up-down symmetry

The limiter and wall geometries are verified to be symmetric about the midplane ($Z=0$).  
Each boundary point $(R,Z)$ is reflected to $(R,-Z)$, and the nearest reflected point is identified. 
The test verifies that every point has a matching reflected counterpart within numerical tolerance and that the nearest-neighbour mapping is one-to-one.

In [17]:
for boundary in (prepared.limiter_data, prepared.wall_data):
    points = np.asarray([[entry["R"], entry["Z"]] for entry in boundary])
    reflected = points.copy()
    reflected[:, 1] *= -1

    distances = np.linalg.norm(
        points[:, np.newaxis] - reflected[np.newaxis, :],
        axis=-1,
    )

    nearest_indices = np.argmin(distances, axis=1)
    nearest_distances = np.min(distances, axis=1)

    np.testing.assert_allclose(nearest_distances, 0.0, atol=1e-14)

    # Every reflected point must be matched exactly once
    assert len(np.unique(nearest_indices)) == len(points)


In [18]:
components = build_tokamak_components(
    active_coils_data=prepared.even_active_coils_data,
    passive_coils_data=prepared.passive_coils_data,
    limiter_data=prepared.limiter_data,
    wall_data=prepared.wall_data,
)
assert components["n_active_coils"] == len(prepared.even_active_names)
assert components["n_passive_coils"] == len(prepared.passive_coils_data)
assert components["n_coils"] == (
    components["n_active_coils"] + components["n_passive_coils"]
)

Active coils --> built from user-provided data.
Passive structures --> built from user-provided data.
Limiter --> built from user-provided data.
Wall --> built from user-provided data.
Magnetic probes --> none provided.
